# Lesson 8 — Make it talk (decoding strategies)

The model from lesson 7 only gives a probability for each next character. *How* I pick from that distribution
changes the output a lot. Three knobs:
- **temperature** — divide logits by T: T < 1 = sharper/safer, T > 1 = flatter/wilder
- **top-k** — keep only the k most likely characters
- **top-p (nucleus)** — keep the smallest set whose probabilities add up to p

In [1]:
import torch, torch.nn as nn, torch.nn.functional as F
torch.manual_seed(1337)

## Same data and config as lesson 7
These must match exactly, or `gpt.pt` won't load into the model.

In [2]:
text = open("shakespeare.txt").read()
chars = sorted(set(text)); vocab_size = len(chars)
stoi = {c: i for i, c in enumerate(chars)}; itos = {i: c for c, i in stoi.items()}
encode = lambda s: [stoi[c] for c in s]
decode = lambda ids: "".join(itos[i] for i in ids)
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data)); train_data, val_data = data[:n], data[n:]

# --- model config (small, so it trains on a CPU in a couple of minutes) ---
block_size = 32     # context length
n_embd     = 64     # embedding size
n_heads    = 4      # attention heads
n_layers   = 3      # transformer blocks
batch_size = 16

## Model classes (identical to lesson 7)

In [3]:
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key   = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))
    def forward(self, x):
        B, T, C = x.shape
        k, q, v = self.key(x), self.query(x), self.value(x)
        wei = q @ k.transpose(-2, -1) * k.shape[-1] ** -0.5
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float("-inf"))
        wei = F.softmax(wei, dim=-1)
        return wei @ v

class MultiHeadAttention(nn.Module):
    def __init__(self, n_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(n_heads)])
        self.proj  = nn.Linear(n_heads * head_size, n_embd)
    def forward(self, x):
        return self.proj(torch.cat([h(x) for h in self.heads], dim=-1))

class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(n_embd, 4 * n_embd), nn.ReLU(), nn.Linear(4 * n_embd, n_embd))
    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    def __init__(self, n_embd, n_heads):
        super().__init__()
        self.sa  = MultiHeadAttention(n_heads, n_embd // n_heads)
        self.ff  = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd); self.ln2 = nn.LayerNorm(n_embd)
    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x

class GPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding    = nn.Embedding(vocab_size, n_embd)
        self.position_embedding = nn.Embedding(block_size, n_embd)
        self.blocks  = nn.Sequential(*[Block(n_embd, n_heads) for _ in range(n_layers)])
        self.ln_f    = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)
    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.token_embedding(idx) + self.position_embedding(torch.arange(T))
        x = self.ln_f(self.blocks(x))
        logits = self.lm_head(x)                       # (B, T, vocab)
        if targets is None:
            return logits, None
        loss = F.cross_entropy(logits.view(B * T, -1), targets.view(B * T))
        return logits, loss
    @torch.no_grad()
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, _ = self(idx[:, -block_size:])
            probs = F.softmax(logits[:, -1, :], dim=-1)
            idx = torch.cat([idx, torch.multinomial(probs, 1)], dim=1)
        return idx

## Load the trained weights
No training here — just the saved `state_dict` from lesson 7. `eval()` switches to inference mode.

In [4]:
model = GPT(); model.load_state_dict(torch.load("gpt.pt")); model.eval()

GPT(
  (token_embedding): Embedding(65, 64)
  (position_embedding): Embedding(32, 64)
  (blocks): Sequential(
    (0): Block(
      (sa): MultiHeadAttention(
        (heads): ModuleList(
          (0-3): 4 x Head(
            (key): Linear(in_features=64, out_features=16, bias=False)
            (query): Linear(in_features=64, out_features=16, bias=False)
            (value): Linear(in_features=64, out_features=16, bias=False)
          )
        )
        (proj): Linear(in_features=64, out_features=64, bias=True)
      )
      (ff): FeedForward(
        (net): Sequential(
          (0): Linear(in_features=64, out_features=256, bias=True)
          (1): ReLU()
          (2): Linear(in_features=256, out_features=64, bias=True)
        )
      )
      (ln1): LayerNorm((64,), eps=1e-05, elementwise_affine=True, bias=True)
      (ln2): LayerNorm((64,), eps=1e-05, elementwise_affine=True, bias=True)
    )
    (1): Block(
      (sa): MultiHeadAttention(
        (heads): ModuleList(
         

## A sampler with all three knobs
- temperature: `logits / temperature`
- top-k: find the k-th largest logit, set everything below it to `-inf`
- top-p: sort, take the running probability total, drop tokens once it passes p (shifted by one so the token that
  crosses p is kept, and the top token is always kept), then scatter back to the original order

Each call reseeds (`seed=0`), so the different settings are compared on the same random draws.

In [5]:
@torch.no_grad()
def sample(max_new_tokens=300, temperature=1.0, top_k=None, top_p=None, seed=0):
    torch.manual_seed(seed)
    idx = torch.zeros((1, 1), dtype=torch.long)
    for _ in range(max_new_tokens):
        logits, _ = model(idx[:, -block_size:])
        logits = logits[:, -1, :] / temperature
        if top_k is not None:
            kth = torch.topk(logits, top_k)[0][:, [-1]]
            logits = logits.masked_fill(logits < kth, float("-inf"))
        if top_p is not None:
            s_logits, s_idx = torch.sort(logits, descending=True)
            cum = F.softmax(s_logits, dim=-1).cumsum(-1)
            drop = cum > top_p; drop[:, 1:] = drop[:, :-1].clone(); drop[:, 0] = False
            s_logits = s_logits.masked_fill(drop, float("-inf"))
            logits = torch.empty_like(logits).scatter_(-1, s_idx, s_logits)
        probs = F.softmax(logits, dim=-1)
        idx = torch.cat([idx, torch.multinomial(probs, 1)], dim=1)
    return decode(idx[0].tolist())

## Low temperature (0.5)

In [6]:
outs = {}
outs["t0.5"] = sample(temperature=0.5)
print("=== temperature 0.5 (conservative) ===\n", outs["t0.5"])

=== temperature 0.5 (conservative) ===
 
thy thou she wisell it of the hat youd thelst tou courese hit therere he he his bere offor on ist that of tor it bee and this couder non my my tood ton you for mandiond the cout cond soud the couse ofor to for his mely my hond sour touse meld for my ind the me he ton sit in be then ast cond to my ha


## High temperature, top-k and top-p

In [7]:
outs["t1.4"] = sample(temperature=1.4)
outs["k10"]  = sample(top_k=10)
outs["p0.9"] = sample(top_p=0.9)
print("=== temperature 1.4 (wild) ===\n", outs["t1.4"])
print("\n=== top-k = 10 ===\n", outs["k10"])
print("\n=== top-p = 0.9 (nucleus) ===\n", outs["p0.9"])

=== temperature 1.4 (wild) ===
 
ty remhis khaw selte, oucay eamay youd Igelst, grecooudsy hitnty
here hath Dot seeiell! mansericest.
-Nory wreit Myemaccesen;
JUforr no- my fittoor epe you fori'd Iioedvotg,
Prescon' souz thul ne.Sevack, pradithid-mily mooe,n, siay,
izecodeld fleome inl' i'dee he tonc.
PKin,bo youn qon gor evexmysha

=== top-k = 10 ===
 
Thy, mhis thereselt ild canderave woud woulste to courese hit therere hath thios ore is sand him;
That of tor it bee ances idy thomall a my my toor spe sto of im whisertthind touc ands sweere crese ouctyend,
Thand mily mooven, sing tint of mevellome in
A thime cencings to bod, then asticge sto my
ha

=== top-p = 0.9 (nucleus) ===
 
thy, mome ther selt ild canderake youd woulste noncomeds,
O ford here hath thios orell soudsericest.

Nor por it Myemacces isart, to non my fittoor epe you for mad iservod.

PEDY:
Hros she you nou for my pradith and loom,
hond sily eigs of levellome int be cee he toncigh, bod, you,
Ton gor hat my ha


## Measuring the differences
Two rough numbers per setting: how many distinct characters are used, and what fraction of "words" are real words
that appear in the training text. Low temperature should use few characters, high temperature many.

In [8]:
vocab_words = set(text.lower().split())
print(f"{'setting':8s} {'distinct chars':>15s} {'real-word share':>16s}")
for name, s in outs.items():
    words = s.lower().split()
    real = sum(w.strip(".,;:!?'-") in vocab_words for w in words) / max(len(words), 1)
    print(f"{name:8s} {len(set(s)):15d} {real:16.0%}")

setting   distinct chars  real-word share
t0.5                  20              59%
t1.4                  42              14%
k10                   27              35%
p0.9                  36              40%


## Reference check against the course output
My `gpt.pt` differs from the course's by tiny rounding amounts (lesson 7 loss curve within 0.002), so the samples
can't be identical: one slightly different probability flips one sampled character, and every later character
depends on it. Below: the start of each course sample next to mine, and how many characters they share.

In [9]:
course_starts = {
    "t0.5": "\nthy thou she wisell it of the hat youd thelst tou couse you thave and he he his be hase",
    "t1.4": "\ntly, mhis khaw selte, oucay eamay youd Igelst, grecourewy hitnty",
    "k10":  "\nThy lorke ther selt ild canderave woud wotoste trecoures,",
    "p0.9": "\nthy forke ther selt ild canderman youd whelst, grecoures,",
}
for name, c in course_starts.items():
    m = outs[name]
    same = next((i for i, (a, b) in enumerate(zip(m, c)) if a != b), len(c))
    print(f"{name:5s} shared prefix: {same:2d} chars")
    print(f"   course: {c.strip()[:60]!r}")
    print(f"   mine  : {m.strip()[:60]!r}")
print("speaker tags in my samples:", [ln for s in outs.values() for ln in s.splitlines() if ln.isupper() and ln.endswith(":")])

t0.5  shared prefix: 54 chars
   course: 'thy thou she wisell it of the hat youd thelst tou couse you '
   mine  : 'thy thou she wisell it of the hat youd thelst tou courese hi'
t1.4  shared prefix:  2 chars
   course: 'tly, mhis khaw selte, oucay eamay youd Igelst, grecourewy hi'
   mine  : 'ty remhis khaw selte, oucay eamay youd Igelst, grecooudsy hi'
k10   shared prefix:  4 chars
   course: 'Thy lorke ther selt ild canderave woud wotoste trecoures,'
   mine  : 'Thy, mhis thereselt ild canderave woud woulste to courese hi'
p0.9  shared prefix:  4 chars
   course: 'thy forke ther selt ild canderman youd whelst, grecoures,'
   mine  : 'thy, mome ther selt ild canderake youd woulste noncomeds,\nO '
speaker tags in my samples: ['PEDY:']


**Behaviour matches the course:**
- **T = 0.5** stays lowercase on one long line and repeats safe words ("my my", "the", "he he his"), the same looping
  as the course's. Only 20 distinct characters, but the highest real-word share (59%).
- **T = 1.4** invents spellings and odd capitals/punctuation (`Iioedvotg`, `PKin,bo`): 42 distinct characters, only
  14% real words.
- **top-k = 10 / top-p = 0.9** sit in between. They're fluent-looking with line breaks and capitalised lines, and
  my top-p sample even produced a speaker tag (`PEDY:`).

The first few characters agree with the course (up to 54 for T = 0.5) before drifting, which is what tiny weight
differences should cause. Interesting detail: after drifting, the texts often **re-sync**. My T = 1.4 sample
differs at character 2 but then has `khaw selte, oucay eamay youd Igelst` exactly like the course. Both runs use the
same seeded random numbers, and once the last few characters agree again the model makes the same picks.

## Test yourself

**1. What does temperature < 1 do to the distribution, and what does the text look like?**
Dividing logits by a number below 1 stretches the gaps between them, so the top characters get even more
probability. The text plays it safe: common words like "the", "my", "he", and it repeats itself.

**2. How is top-p different from top-k?**
Top-k always keeps exactly k tokens. Top-p keeps however many it takes to cover probability p: few when the model is
confident, more when it's unsure. So it adapts to each step.

**3. Why does the top-p code shift `drop` by one position?**
So the token that pushes the running total over p is still kept, along with the top token. Otherwise there could be
nothing left to sample.

## My notes
- The model is fixed; decoding settings alone move the output from repetitive to gibberish. Same knobs as a chat API.
- Surprising: my samples drift from the course's after a few characters and then snap back into the same words, because the random draws are shared.